# RAGAS Evaluation Notebook - Single-Turn and Multi-Turn

This merged notebook evaluates the three Excel inputs only:

1. Easy Travel single-turn sheet
2. PolicyBot single-turn sheet
3. PolicyBot multi-turn sheet

The flow is: load shared setup, configure Azure OpenAI once, upload the three sheets once, run single-turn predefined metrics, run single-turn custom metrics, then run the two multi-turn SimpleCriteriaScore metrics.

## 1. Install dependencies

Run this once per session. If the packages are already installed, you can skip this cell.

In [ ]:
%pip install \
    ragas==0.3.9 \
    langchain==0.3.27 \
    langchain-openai==0.3.28 \
    langchain-community==0.3.27 \
    langchain-huggingface==0.1.2 \
    openai==1.99.5 \
    datasets==4.0.0 \
    pandas==2.2.2 \
    sentence-transformers==3.4.1 \
    nest_asyncio==1.6.0 \
    openpyxl==3.1.5
print("Install complete. If running on Colab, restart the runtime before continuing.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.7/366.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 32.3 MB/s

Install complete. If running on Colab, restart the runtime before continuing.


## 2. Imports and shared helpers

These helpers load Excel workbooks, normalize common column names, and build RAGAS datasets from the three uploaded sheets. They do not contain sample data.

In [ ]:
import ast
import os
from pathlib import Path

import nest_asyncio
import pandas as pd
from IPython.display import display

from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.dataset_schema import MultiTurnSample
from ragas.messages import HumanMessage, AIMessage

nest_asyncio.apply()

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

SINGLE_TURN_COLUMN_ALIASES = {
    "user_input": ["question", "user_input", "input", "query", "prompt"],
    "reference": ["ground_truth", "groundtruth", "reference", "expected_output", "expected", "gold_answer", "expected_answer"],
    "response": ["answer", "response", "actual_output", "output", "prediction", "model_answer", "actual_answer", "generated_output"],
    "retrieved_contexts": ["contexts", "context", "retrieved_contexts", "retrieval_context", "retrieved_context", "actual_context", "actual_contex", "actual_contexts"],
}
CONTEXT_DELIMITERS = ["||", "\n", ";"]

MULTI_TURN_COLUMN_ALIASES = {
    "input": "question",
    "user_input": "question",
    "prompt": "question",
    "actual_output": "actual_answer",
    "generated_output": "actual_answer",
    "output": "actual_answer",
    "response": "actual_answer",
    "answer": "actual_answer",
    "expected_output": "expected_answer",
    "expected": "expected_answer",
    "reference": "expected_answer",
    "ground_truth": "expected_answer",
}


def _norm(name):
    return str(name).strip().lower().replace(" ", "_")


def _is_blank(value):
    return value is None or (isinstance(value, float) and pd.isna(value)) or str(value).strip() == ""


def _text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def upload_excel(label):
    print(f"Upload the {label} Excel file (.xlsx)")
    if IN_COLAB:
        uploaded = files.upload()
        excel_path = next(iter(uploaded))
    else:
        excel_path = input(f"Path to the {label} Excel file: ").strip().strip('"')
    if not excel_path:
        raise ValueError(f"No Excel file was provided for {label}.")
    df = pd.read_excel(excel_path)
    print(f'Loaded "{excel_path}" with {len(df)} row(s).')
    print("Columns found:", list(df.columns))
    return excel_path, df


def build_single_turn_field_map(columns):
    norm_cols = {_norm(c): c for c in columns}
    field_map = {}
    for field, aliases in SINGLE_TURN_COLUMN_ALIASES.items():
        for alias in aliases:
            if alias in norm_cols:
                field_map[field] = norm_cols[alias]
                break
    return field_map


def split_contexts(value):
    if _is_blank(value):
        return None
    text = str(value).strip()
    for delim in CONTEXT_DELIMITERS:
        if delim in text:
            parts = [part.strip() for part in text.split(delim) if part.strip()]
            return parts or None
    return [text]


def excel_to_single_turn_dataset(df, label):
    field_map = build_single_turn_field_map(df.columns)
    print(f"\n{label}: detected column -> RAGAS field")
    for field, col in field_map.items():
        print(f"  {col!r} -> {field}")

    required = {"user_input", "response"}
    missing_required = required - set(field_map)
    if missing_required:
        raise ValueError(f"{label} is missing required single-turn field(s): {sorted(missing_required)}")

    samples = []
    for _, row in df.iterrows():
        kwargs = {}
        for field, col in field_map.items():
            value = row.get(col)
            if field == "retrieved_contexts":
                contexts = split_contexts(value)
                if contexts is not None:
                    kwargs[field] = contexts
            elif not _is_blank(value):
                kwargs[field] = str(value).strip()
        if kwargs:
            samples.append(SingleTurnSample(**kwargs))

    if not samples:
        raise ValueError(f"No usable single-turn samples were found in {label}.")

    print(f"Built {len(samples)} single-turn sample(s) from {len(df)} row(s).")
    return EvaluationDataset(samples=samples), set(field_map)


def normalize_multi_turn_columns(df):
    normalized = df.copy()
    normalized.columns = [_norm(col) for col in normalized.columns]
    rename_map = {
        source: target
        for source, target in MULTI_TURN_COLUMN_ALIASES.items()
        if source in normalized.columns and target not in normalized.columns
    }
    return normalized.rename(columns=rename_map)



def parse_ragas_messages(value):
    text = _text(value)
    if not text:
        return []

    tree = ast.parse(text, mode="eval")
    if not isinstance(tree.body, (ast.List, ast.Tuple)):
        raise ValueError("The ragas cell must be a list of HumanMessage(...) and AIMessage(...).")

    messages = []
    for node in tree.body.elts:
        if not isinstance(node, ast.Call) or not isinstance(node.func, ast.Name):
            raise ValueError("Each ragas entry must be HumanMessage(...) or AIMessage(...).")
        if node.func.id not in {"HumanMessage", "AIMessage"}:
            raise ValueError(f"Unsupported ragas message type: {node.func.id}")

        kwargs = {kw.arg: kw.value for kw in node.keywords if kw.arg}
        content_node = kwargs.get("content")
        if content_node is None and node.args:
            content_node = node.args[0]
        if not isinstance(content_node, ast.Constant) or not isinstance(content_node.value, str):
            raise ValueError("Each ragas message must have a string content value.")

        message_cls = HumanMessage if node.func.id == "HumanMessage" else AIMessage
        messages.append(message_cls(content=content_node.value))

    return messages


def message_role(message):
    if isinstance(message, HumanMessage):
        return "User"
    if isinstance(message, AIMessage):
        return "Assistant"
    return type(message).__name__


def messages_to_text(messages):
    lines = []
    for index, message in enumerate(messages, start=1):
        lines.append(f"{index}. {message_role(message)}: {message.content}")
    return "\n".join(lines)


def build_reference(group):
    if "expected_answer" not in group.columns:
        return ""
    parts = []
    for _, row in group.iterrows():
        expected = _text(row.get("expected_answer"))
        if not expected:
            continue
        turn = _text(row.get("turn")) if "turn" in group.columns else ""
        question = _text(row.get("question")) if "question" in group.columns else ""
        label = f"Turn {turn}" if turn else "Turn"
        if question:
            parts.append(f"{label}\nUser: {question}\nExpected assistant answer: {expected}")
        else:
            parts.append(f"{label}\nExpected assistant answer: {expected}")
    return "\n\n".join(parts)


def build_actual_trace(group):
    parts = []
    for _, row in group.iterrows():
        question = _text(row.get("question")) if "question" in group.columns else ""
        actual = _text(row.get("actual_answer")) if "actual_answer" in group.columns else ""
        turn = _text(row.get("turn")) if "turn" in group.columns else ""
        if not question and not actual:
            continue
        label = f"Turn {turn}" if turn else "Turn"
        lines = [label]
        if question:
            lines.append(f"User: {question}")
        if actual:
            lines.append(f"Assistant: {actual}")
        parts.append("\n".join(lines))
    return "\n\n".join(parts)


def build_messages_from_rows(group):
    messages = []
    for _, row in group.iterrows():
        question = _text(row.get("question")) if "question" in group.columns else ""
        actual = _text(row.get("actual_answer")) if "actual_answer" in group.columns else ""
        if question:
            messages.append(HumanMessage(content=question))
        if actual:
            messages.append(AIMessage(content=actual))
    return messages


def excel_to_multi_turn_dataset(df, label):
    raw_df = normalize_multi_turn_columns(df)
    if not ({"ragas"} & set(raw_df.columns) or {"question", "actual_answer"}.issubset(raw_df.columns)):
        raise ValueError(
            f"{label} must contain a 'ragas' column, or at least 'question' and 'actual_answer' columns."
        )

    if "case_id" not in raw_df.columns:
        raw_df["case_id"] = [f"ROW-{idx:04d}" for idx in range(len(raw_df))]
    if "turn" in raw_df.columns:
        raw_df["turn_sort"] = pd.to_numeric(raw_df["turn"], errors="coerce")
    else:
        raw_df["turn_sort"] = raw_df.groupby("case_id").cumcount() + 1

    raw_df = raw_df.sort_values(["case_id", "turn_sort"], kind="stable")

    samples = []
    input_rows = []
    for case_id, group in raw_df.groupby("case_id", sort=False):
        ragas_messages = []
        ragas_source_turn = ""
        if "ragas" in group.columns:
            # Use the last non-empty ragas cell in the case. In these sheets, that is
            # normally the final row containing the complete conversation.
            for _, row in group.iterrows():
                if _text(row.get("ragas")):
                    ragas_messages = parse_ragas_messages(row.get("ragas"))
                    ragas_source_turn = _text(row.get("turn")) if "turn" in group.columns else ""

        row_messages = build_messages_from_rows(group)
        messages = ragas_messages or row_messages
        if not messages:
            continue

        reference = build_reference(group)
        actual_trace = build_actual_trace(group)
        user_input_text = messages_to_text(messages)
        sample_kwargs = {"user_input": messages}
        if reference:
            sample_kwargs["reference"] = reference

        samples.append(MultiTurnSample(**sample_kwargs))
        input_rows.append(
            {
                "case_id": case_id,
                "turn_count": len(group),
                "message_count": len(messages),
                "source": "ragas" if ragas_messages else "question_actual_answer",
                "ragas_source_turn": ragas_source_turn,
                "has_reference": bool(reference),
                "user_input": user_input_text,
                "actual_answer_trace": actual_trace,
                "reference": reference,
            }
        )

    if not samples:
        raise ValueError(f"No usable multi-turn conversations were found in {label}.")

    parsed_inputs_df = pd.DataFrame(input_rows)
    print(f"Built {len(samples)} multi-turn conversation sample(s).")
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(parsed_inputs_df)
    return EvaluationDataset(samples=samples), parsed_inputs_df



def parse_ragas_user_input(ragas_value):
    """Parse one sheet ragas cell into RAGAS HumanMessage/AIMessage objects."""
    ragas_text = _text(ragas_value)
    if not ragas_text:
        return []

    tree = ast.parse(ragas_text, mode="eval")
    if not isinstance(tree.body, (ast.List, ast.Tuple)):
        raise ValueError("Each non-empty ragas cell must contain a list of HumanMessage(...) and AIMessage(...).")

    messages = []
    for node in tree.body.elts:
        if not isinstance(node, ast.Call) or not isinstance(node.func, ast.Name):
            raise ValueError("Each ragas list item must be HumanMessage(...) or AIMessage(...).")
        if node.func.id not in {"HumanMessage", "AIMessage"}:
            raise ValueError(f"Unsupported message type in ragas column: {node.func.id}")

        kwargs = {kw.arg: kw.value for kw in node.keywords if kw.arg}
        content_node = kwargs.get("content")
        if content_node is None and node.args:
            content_node = node.args[0]
        if not isinstance(content_node, ast.Constant) or not isinstance(content_node.value, str):
            raise ValueError("Each HumanMessage/AIMessage must have string content.")

        message_cls = HumanMessage if node.func.id == "HumanMessage" else AIMessage
        messages.append(message_cls(content=content_node.value))

    return messages


def format_multiturn_messages(messages):
    lines = []
    for index, message in enumerate(messages, start=1):
        role = "User" if isinstance(message, HumanMessage) else "Assistant"
        lines.append(f"{index}. {role}: {message.content}")
    return "\n".join(lines)


def build_reference_from_expected_answers(group):
    parts = []
    for _, row in group.iterrows():
        expected = _text(row.get("expected_answer"))
        if not expected:
            continue
        turn = _text(row.get("turn")) if "turn" in group.columns else ""
        question = _text(row.get("question")) if "question" in group.columns else ""
        label = f"Turn {turn}" if turn else "Turn"
        if question:
            parts.append(f"{label}\nUser: {question}\nExpected assistant answer: {expected}")
        else:
            parts.append(f"{label}\nExpected assistant answer: {expected}")
    return "\n\n".join(parts)


def build_multiturn_ragas_datasets(df, label="PolicyBot multi-turn"):
    """Use the final non-empty ragas cell per case_id as the exact MultiTurnSample user_input."""
    raw_df = df.copy()
    raw_df.columns = [_norm(col) for col in raw_df.columns]

    required_columns = {"case_id", "ragas", "expected_answer"}
    missing_columns = required_columns - set(raw_df.columns)
    if missing_columns:
        raise ValueError(f"{label} sheet is missing required column(s): {sorted(missing_columns)}")

    if "turn" in raw_df.columns:
        raw_df["turn_sort"] = pd.to_numeric(raw_df["turn"], errors="coerce")
    else:
        raw_df["turn_sort"] = raw_df.groupby("case_id").cumcount() + 1

    raw_df = raw_df.sort_values(["case_id", "turn_sort"], kind="stable")

    samples_without_reference = []
    samples_with_reference = []
    reference_row_indexes = []
    parsed_rows = []

    for case_id, group in raw_df.groupby("case_id", sort=False):
        non_empty_ragas_rows = group[group["ragas"].apply(lambda value: bool(_text(value)))]
        if non_empty_ragas_rows.empty:
            print(f"Skipping {case_id}: no non-empty ragas cell found.")
            continue

        ragas_row = non_empty_ragas_rows.iloc[-1]
        messages = parse_ragas_user_input(ragas_row["ragas"])
        if not messages:
            print(f"Skipping {case_id}: ragas cell parsed to no messages.")
            continue

        reference = build_reference_from_expected_answers(group)
        parsed_row_index = len(parsed_rows)
        samples_without_reference.append(MultiTurnSample(user_input=messages))
        if reference:
            samples_with_reference.append(MultiTurnSample(user_input=messages, reference=reference))
            reference_row_indexes.append(parsed_row_index)

        parsed_rows.append(
            {
                "case_id": case_id,
                "ragas_source_turn": ragas_row.get("turn", ""),
                "message_count": len(messages),
                "has_reference": bool(reference),
                "user_input": format_multiturn_messages(messages),
                "reference": reference,
            }
        )

    parsed_df = pd.DataFrame(parsed_rows)
    if parsed_df.empty:
        raise ValueError(f"No usable multi-turn samples were parsed from the {label} ragas column.")

    return {
        "parsed_df": parsed_df,
        "dataset_without_reference": EvaluationDataset(samples=samples_without_reference),
        "dataset_with_reference": EvaluationDataset(samples=samples_with_reference),
        "reference_row_indexes": reference_row_indexes,
        "all_have_reference": len(samples_with_reference) == len(samples_without_reference),
    }


def ragas_score_columns(result_df):
    input_columns = {"user_input", "reference", "reference_topics", "reference_tool_calls"}
    return [col for col in result_df.columns if col not in input_columns]

print("Imports and helpers ready.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Imports and helpers ready.


## 3. Configure Azure OpenAI credentials

This is the only credential/configuration cell in the notebook. Fill in all Azure OpenAI values here once; every metric section reuses these objects.

In [ ]:
import os

AZURE_OPENAI_API_KEY = "FkrroqazACOGKkmt"
AZURE_OPENAI_API_VERSION = "202eview"
AZURE_OPENAI_ENDPOINT = "https:.azure.com/"
AZURE_OPENAI_DEPLOYMENT_NAME = "gpt-5"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

os.environ["AZURE_OPENAI_API_KEY"] = AZURE_OPENAI_API_KEY
os.environ["AZURE_OPENAI_API_VERSION"] = AZURE_OPENAI_API_VERSION
os.environ["AZURE_OPENAI_ENDPOINT"] = AZURE_OPENAI_ENDPOINT
os.environ["AZURE_DEPLOYMENT_NAME"] = AZURE_OPENAI_DEPLOYMENT_NAME

from langchain_openai import AzureChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms.base import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

class GPT5LangchainLLMWrapper(LangchainLLMWrapper):
    def generate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
        return super().generate_text(prompt, n=n, temperature=1, stop=stop, callbacks=callbacks)

    async def agenerate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
        return await super().agenerate_text(prompt, n=n, temperature=1, stop=stop, callbacks=callbacks)

llm_client = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_DEPLOYMENT_NAME,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=1,
)
wrapped_llm = GPT5LangchainLLMWrapper(llm_client, bypass_temperature=True)
evaluator_llm = wrapped_llm

evaluator_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
)

print("API key present?", bool(AZURE_OPENAI_API_KEY and AZURE_OPENAI_API_KEY != "YOUR_AZURE_OPENAI_API_KEY"))
print("Endpoint:", AZURE_OPENAI_ENDPOINT)
print("Deployment:", AZURE_OPENAI_DEPLOYMENT_NAME)
print("RAGAS LLM and embeddings ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

API key present? True
Endpoint: https://ai-assurance-enablement-temp.openai.azure.com/
Deployment: gpt-5
RAGAS LLM and embeddings ready.


/tmp/ipykernel_748/2874058866.py:36: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(


## 4. Upload Easy Travel single-turn sheet

In [ ]:
easytravel_single_turn_path, easytravel_single_turn_df = upload_excel("Easy Travel single-turn")
easytravel_single_turn_dataset, easytravel_single_turn_fields = excel_to_single_turn_dataset(
    easytravel_single_turn_df,
    "Easy Travel single-turn",
)

Upload the Easy Travel single-turn Excel file (.xlsx)


Saving EasyTravel_Sample_Transcripts.xlsx to EasyTravel_Sample_Transcripts.xlsx
Loaded "EasyTravel_Sample_Transcripts.xlsx" with 10 row(s).
Columns found: ['Scenario', 'Input', 'Expected_Output', 'Actual_Output']

Easy Travel single-turn: detected column -> RAGAS field
  'Input' -> user_input
  'Expected_Output' -> reference
  'Actual_Output' -> response
Built 10 single-turn sample(s) from 10 row(s).


## 5. Upload PolicyBot single-turn sheet

In [ ]:
policybot_single_turn_path, policybot_single_turn_df = upload_excel("PolicyBot single-turn")
policybot_single_turn_dataset, policybot_single_turn_fields = excel_to_single_turn_dataset(
    policybot_single_turn_df,
    "PolicyBot single-turn",
)

Upload the PolicyBot single-turn Excel file (.xlsx)


Saving PolicyBot_Sample_QA.xlsx to PolicyBot_Sample_QA.xlsx
Loaded "PolicyBot_Sample_QA.xlsx" with 20 row(s).
Columns found: ['Input', 'Expected_Output', 'Expected_Context', 'Actual_Output', 'Actual_Context']

PolicyBot single-turn: detected column -> RAGAS field
  'Input' -> user_input
  'Expected_Output' -> reference
  'Actual_Output' -> response
  'Actual_Context' -> retrieved_contexts
Built 20 single-turn sample(s) from 20 row(s).


## 6. Upload PolicyBot multi-turn sheet

In [ ]:
policybot_multi_turn_path, policybot_multi_turn_df = upload_excel("PolicyBot multi-turn")

Upload the PolicyBot multi-turn Excel file (.xlsx)


Saving policybot_multiturn_output_generation.xlsx to policybot_multiturn_output_generation.xlsx
Loaded "policybot_multiturn_output_generation.xlsx" with 16 row(s).
Columns found: ['case_id', 'turn', 'question', 'expected_answer', 'actual_answer', 'deepeval', 'ragas']


# Single-Turn Evaluation

## 7. Single-turn predefined metrics

Easy Travel uses no-context metrics. PolicyBot uses the full RAG metric set when the sheet includes retrieved contexts.

In [ ]:
from ragas.metrics import (
    AnswerCorrectness,
    AnswerSimilarity,
    Faithfulness,
    ContextRecall,
    ContextPrecision,
    NoiseSensitivity,
)

answer_similarity = AnswerSimilarity(embeddings=evaluator_embeddings)

single_turn_no_context_metrics = [
    AnswerCorrectness(llm=wrapped_llm, answer_similarity=answer_similarity),
    answer_similarity,
]

single_turn_rag_metrics = [
    AnswerCorrectness(llm=wrapped_llm, answer_similarity=answer_similarity),
    answer_similarity,
    Faithfulness(llm=wrapped_llm),
    ContextRecall(llm=wrapped_llm),
    ContextPrecision(llm=wrapped_llm),
    NoiseSensitivity(llm=wrapped_llm),
]

policybot_single_turn_predefined_metrics = (
    single_turn_rag_metrics
    if "retrieved_contexts" in policybot_single_turn_fields
    else single_turn_no_context_metrics
)

print("Easy Travel predefined metrics:", [getattr(metric, "name", metric.__class__.__name__) for metric in single_turn_no_context_metrics])
print("PolicyBot predefined metrics:", [getattr(metric, "name", metric.__class__.__name__) for metric in policybot_single_turn_predefined_metrics])

Easy Travel predefined metrics: ['answer_correctness', 'answer_similarity']
PolicyBot predefined metrics: ['answer_correctness', 'answer_similarity', 'faithfulness', 'context_recall', 'context_precision', 'noise_sensitivity']


In [ ]:
easytravel_single_turn_predefined_result = evaluate(
    dataset=easytravel_single_turn_dataset,
    metrics=single_turn_no_context_metrics,
    llm=wrapped_llm,
    embeddings=evaluator_embeddings,
)
easytravel_single_turn_predefined_df = easytravel_single_turn_predefined_result.to_pandas()
display(easytravel_single_turn_predefined_df)
easytravel_single_turn_predefined_df.to_excel("easytravel_single_turn_predefined_ragas_results.xlsx", index=False)
print("Saved easytravel_single_turn_predefined_ragas_results.xlsx")

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

,user_input,response,reference,answer_correctness,answer_similarity
0,"Agent: Thank you for calling EasyTravel, this ...",Customer Rohan Mehta requested cancellation of...,Customer Rohan Mehta requested cancellation of...,1.000000,1.000000
1,"Agent: Good afternoon, EasyTravel support, Rah...",Customer Anita Sharma (Silver member) requeste...,Customer Anita Sharma (Gold member) requested ...,0.410935,0.825558
2,"Agent: EasyTravel support, this is Ananya. How...",Customer Vikram Iyer reported missing checked-...,Customer Vikram Iyer reported missing checked-...,1.000000,1.000000
3,"Agent: EasyTravel customer service, this is Ka...",Customer Meena Joseph's travel insurance claim...,Customer Meena Joseph has been chasing travel ...,0.357397,0.829586
4,"Agent: Hello, EasyTravel loyalty support, this...","Customer Arjun Kapoor wanted to redeem 65,000 ...","Customer Arjun Kapoor wanted to redeem 65,000 ...",1.000000,1.000000
5,"Agent: EasyTravel hotels, you have Divya speak...",Customer Sameer Khan amended hotel booking ETH...,Customer Sameer Khan amended hotel booking ETH...,0.960526,1.000000
6,"Agent: EasyTravel group desk, this is Pooja sp...",Corporate group of 15 from Lumen Edutech for B...,Corporate group of 18 from Lumen Edutech for B...,0.313120,0.752479
7,"Agent: EasyTravel, this is Aakash. How may I a...",Customer Reema Pillai requested rescheduling o...,Customer Reema Pillai requested rescheduling o...,1.000000,1.000000
8,"Agent: EasyTravel billing, you have Naveen on ...",Customer Tanvi Bhatt reported a duplicate Rs. ...,Customer Tanvi Bhatt reported a duplicate Rs. ...,0.405662,0.872647
9,"Agent: Welcome to EasyTravel holidays, you hav...",Customer Karthik Reddy booked the Maldives Hon...,Customer Karthik Reddy booked the Maldives Hon...,1.000000,1.000000


Saved easytravel_single_turn_predefined_ragas_results.xlsx


In [ ]:
policybot_single_turn_predefined_result = evaluate(
    dataset=policybot_single_turn_dataset,
    metrics=policybot_single_turn_predefined_metrics,
    llm=wrapped_llm,
    embeddings=evaluator_embeddings,
)
policybot_single_turn_predefined_df = policybot_single_turn_predefined_result.to_pandas()
display(policybot_single_turn_predefined_df)
policybot_single_turn_predefined_df.to_excel("policybot_single_turn_predefined_ragas_results.xlsx", index=False)
print("Saved policybot_single_turn_predefined_ragas_results.xlsx")

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,answer_correctness,answer_similarity,faithfulness,context_recall,context_precision,noise_sensitivity(mode=relevant)
0,How many earned leaves can an employee carry f...,"[As per the leave policy, unused Earned Leave ...",Employees can carry forward only up to 25 days...,Up to 30 days of EL can be carried forward; ba...,0.158781,0.635122,0.500000,0.0,0.0,0.000000
1,What is the per diem allowance for internation...,[International per diem: USD 75 per day for tr...,The per diem allowance is USD 75 per day for t...,USD 75/day for USA/UK/Western Europe/Singapore...,0.894445,0.577780,1.000000,1.0,1.0,0.000000
2,How many dependents are covered under medical ...,[An employee may enroll a maximum of 4 depende...,A maximum of 4 dependents can be covered under...,Maximum 2 dependents under the base GMC policy...,0.660513,0.927765,1.000000,0.0,0.0,0.500000
3,"Who approves expense claims above Rs. 50,000?","[Expense claims above Rs. 50,000 require appro...","Expense claims above Rs. 50,000 are approved b...",The relevant Vice President approves claims ab...,0.167807,0.671229,1.000000,0.0,0.0,0.000000
4,Can an employee on probation avail WFH?,[All confirmed employees of Meridian Technolog...,"No, probationers are not eligible for standard...",Probationers are NOT eligible for standard WFH...,0.986332,0.945326,1.000000,1.0,1.0,0.000000
5,What is the maximum number of WFH days per wee...,[Confirmed employees may avail a maximum of 2 ...,Confirmed employees can avail a maximum of 2 W...,Maximum 2 WFH days per week.,0.966787,0.867148,1.000000,1.0,1.0,1.000000
6,How is the annual performance rating calculated?,[Final ratings are computed as a weighted scor...,The annual rating is weighted at 60% for goal ...,Weighted: 70% goal achievement and 30% behavio...,0.601416,0.905665,0.000000,0.0,0.0,0.000000
7,What documents are required for medical reimbu...,"[Original receipts, discharge summary, and the...","You must submit original receipts, the dischar...","Original receipts, discharge summary, and doct...",0.987051,0.948203,1.000000,1.0,1.0,0.750000
8,What is the policy on software installation on...,[Only software from the IT-approved catalogue ...,Only software from the IT-approved catalogue m...,Only software from the IT-approved catalogue; ...,0.986924,0.947694,0.666667,1.0,1.0,0.000000
9,How many casual leaves can be clubbed with ear...,[Casual Leave cannot be clubbed with Earned Le...,Casual leave cannot be clubbed with earned lea...,CL cannot be clubbed with EL under any circums...,0.912771,0.651084,1.000000,1.0,1.0,0.000000


Saved policybot_single_turn_predefined_ragas_results.xlsx


## 8. Single-turn custom metrics

These AspectCritic metrics are applied directly to the uploaded single-turn sheets.

In [ ]:
from ragas.metrics import AspectCritic

single_turn_answer_correctness_critic = AspectCritic(
    name="answer_correctness_binary",
    definition="Return 1 if the answer correctly conveys the same meaning as the reference answer; otherwise return 0.",
    llm=wrapped_llm,
)

single_turn_fluency = AspectCritic(
    name="fluency_binary",
    definition="Return 1 if the answer is grammatically correct and easy to understand; otherwise return 0.",
    llm=wrapped_llm,
)

single_turn_completeness = AspectCritic(
    name="completeness_binary",
    definition="Return 1 if the answer fully addresses the question using the available reference or context; otherwise return 0.",
    llm=wrapped_llm,
)

single_turn_conciseness = AspectCritic(
    name="conciseness_binary",
    definition="Return 1 if the answer is direct and avoids unnecessary or repeated information; otherwise return 0.",
    llm=wrapped_llm,
)

single_turn_groundedness = AspectCritic(
    name="groundedness_binary",
    definition="Return 1 if every answer claim is supported by the provided context or reference; otherwise return 0.",
    llm=wrapped_llm,
)

single_turn_custom_metrics = [
    single_turn_answer_correctness_critic,
    single_turn_fluency,
    single_turn_completeness,
    single_turn_conciseness,
    single_turn_groundedness,
]
print("Single-turn custom metrics:", [metric.name for metric in single_turn_custom_metrics])

Single-turn custom metrics: ['answer_correctness_binary', 'fluency_binary', 'completeness_binary', 'conciseness_binary', 'groundedness_binary']


In [ ]:
easytravel_single_turn_custom_result = evaluate(
    dataset=easytravel_single_turn_dataset,
    metrics=single_turn_custom_metrics,
    llm=wrapped_llm,
)
easytravel_single_turn_custom_df = easytravel_single_turn_custom_result.to_pandas()
display(easytravel_single_turn_custom_df)
easytravel_single_turn_custom_df.to_excel("easytravel_single_turn_custom_ragas_results.xlsx", index=False)
print("Saved easytravel_single_turn_custom_ragas_results.xlsx")

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

,user_input,response,reference,answer_correctness_binary,fluency_binary,completeness_binary,conciseness_binary,groundedness_binary
0,"Agent: Thank you for calling EasyTravel, this ...",Customer Rohan Mehta requested cancellation of...,Customer Rohan Mehta requested cancellation of...,1,1,1,1,1
1,"Agent: Good afternoon, EasyTravel support, Rah...",Customer Anita Sharma (Silver member) requeste...,Customer Anita Sharma (Gold member) requested ...,0,1,0,1,0
2,"Agent: EasyTravel support, this is Ananya. How...",Customer Vikram Iyer reported missing checked-...,Customer Vikram Iyer reported missing checked-...,1,1,1,1,1
3,"Agent: EasyTravel customer service, this is Ka...",Customer Meena Joseph's travel insurance claim...,Customer Meena Joseph has been chasing travel ...,0,1,0,0,0
4,"Agent: Hello, EasyTravel loyalty support, this...","Customer Arjun Kapoor wanted to redeem 65,000 ...","Customer Arjun Kapoor wanted to redeem 65,000 ...",1,1,1,1,1
5,"Agent: EasyTravel hotels, you have Divya speak...",Customer Sameer Khan amended hotel booking ETH...,Customer Sameer Khan amended hotel booking ETH...,1,1,1,1,1
6,"Agent: EasyTravel group desk, this is Pooja sp...",Corporate group of 15 from Lumen Edutech for B...,Corporate group of 18 from Lumen Edutech for B...,0,0,0,1,0
7,"Agent: EasyTravel, this is Aakash. How may I a...",Customer Reema Pillai requested rescheduling o...,Customer Reema Pillai requested rescheduling o...,1,1,1,1,1
8,"Agent: EasyTravel billing, you have Naveen on ...",Customer Tanvi Bhatt reported a duplicate Rs. ...,Customer Tanvi Bhatt reported a duplicate Rs. ...,0,1,0,1,0
9,"Agent: Welcome to EasyTravel holidays, you hav...",Customer Karthik Reddy booked the Maldives Hon...,Customer Karthik Reddy booked the Maldives Hon...,1,1,1,1,1


Saved easytravel_single_turn_custom_ragas_results.xlsx


In [ ]:
policybot_single_turn_custom_result = evaluate(
    dataset=policybot_single_turn_dataset,
    metrics=single_turn_custom_metrics,
    llm=wrapped_llm,
)
policybot_single_turn_custom_df = policybot_single_turn_custom_result.to_pandas()
display(policybot_single_turn_custom_df)
policybot_single_turn_custom_df.to_excel("policybot_single_turn_custom_ragas_results.xlsx", index=False)
print("Saved policybot_single_turn_custom_ragas_results.xlsx")

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,answer_correctness_binary,fluency_binary,completeness_binary,conciseness_binary,groundedness_binary
0,How many earned leaves can an employee carry f...,"[As per the leave policy, unused Earned Leave ...",Employees can carry forward only up to 25 days...,Up to 30 days of EL can be carried forward; ba...,0,1,0,1,0
1,What is the per diem allowance for internation...,[International per diem: USD 75 per day for tr...,The per diem allowance is USD 75 per day for t...,USD 75/day for USA/UK/Western Europe/Singapore...,1,1,1,1,1
2,How many dependents are covered under medical ...,[An employee may enroll a maximum of 4 depende...,A maximum of 4 dependents can be covered under...,Maximum 2 dependents under the base GMC policy...,0,1,0,1,1
3,"Who approves expense claims above Rs. 50,000?","[Expense claims above Rs. 50,000 require appro...","Expense claims above Rs. 50,000 are approved b...",The relevant Vice President approves claims ab...,0,1,0,1,1
4,Can an employee on probation avail WFH?,[All confirmed employees of Meridian Technolog...,"No, probationers are not eligible for standard...",Probationers are NOT eligible for standard WFH...,1,1,1,1,1
5,What is the maximum number of WFH days per wee...,[Confirmed employees may avail a maximum of 2 ...,Confirmed employees can avail a maximum of 2 W...,Maximum 2 WFH days per week.,1,1,1,1,1
6,How is the annual performance rating calculated?,[Final ratings are computed as a weighted scor...,The annual rating is weighted at 60% for goal ...,Weighted: 70% goal achievement and 30% behavio...,0,1,0,1,1
7,What documents are required for medical reimbu...,"[Original receipts, discharge summary, and the...","You must submit original receipts, the dischar...","Original receipts, discharge summary, and doct...",1,1,1,1,1
8,What is the policy on software installation on...,[Only software from the IT-approved catalogue ...,Only software from the IT-approved catalogue m...,Only software from the IT-approved catalogue; ...,1,1,1,1,1
9,How many casual leaves can be clubbed with ear...,[Casual Leave cannot be clubbed with Earned Le...,Casual leave cannot be clubbed with earned lea...,CL cannot be clubbed with EL under any circums...,1,1,1,1,1


Saved policybot_single_turn_custom_ragas_results.xlsx


# Multi-Turn Evaluation

## 9. Parse PolicyBot multi-turn `ragas` input

This uses the final non-empty `ragas` cell per `case_id` as the exact `MultiTurnSample.user_input` accepted by RAGAS.

In [ ]:
policybot_multi_turn_data = build_multiturn_ragas_datasets(policybot_multi_turn_df, "PolicyBot multi-turn")
policybot_multi_turn_parsed_df = policybot_multi_turn_data["parsed_df"]

print(f"Parsed {len(policybot_multi_turn_parsed_df)} conversation(s) from the ragas column.")
with pd.option_context("display.max_colwidth", None, "display.max_rows", None, "display.max_columns", None):
    display(policybot_multi_turn_parsed_df)

Parsed 3 conversation(s) from the ragas column.


,case_id,ragas_source_turn,message_count,has_reference,user_input,reference
0,MULTI-000,3,6,True,"1. User: How many casual leaves can an employee take in a year?\n2. Assistant: An employee can take 6 casual leaves in a year, as per their entitlement.\n3. User: Can those leaves be carried forward if they are unused?\n4. Assistant: Yes, unused earned leave (EL) can be carried forward to the next year, subject to a maximum of 30 days as per the policy.\n5. User: What should I remember before applying for this leave?\n6. Assistant: Before applying for leave, ensure you submit your request through the HRMS leave module, verify that it is approved before starting your absence, and follow the approval workflow. Also, for leaves of 5 days or more, approval from the Business Unit Head is required.","Turn 1\nUser: How many casual leaves can an employee take in a year?\nExpected assistant answer: An employee can take 6 days of Casual Leave per calendar year.\n\nTurn 2\nUser: Can those leaves be carried forward if they are unused?\nExpected assistant answer: Yes, unused Earned Leave (EL) can be carried forward to the next calendar year, subject to a maximum accumulation of 30 days, as specified in clauses on page 4 of HR_Leave_Policy.pdf.\n\nTurn 3\nUser: What should I remember before applying for this leave?\nExpected assistant answer: Before applying for leave, remember to apply through the HRMS leave module, verify that your leave request has been approved before starting your absence, and note that manual applications are only accepted when HRMS is unavailable and must be regularized within 3 working days; also, approval depends on the workflow of Reporting Manager, Business Unit Head (for EL of 5 days or more), and HR, with the policy reviewed at least once every two years."
1,MULTI-001,5,10,True,"1. User: What types of leave are available to employees?\n2. Assistant: Employees have access to various paid leave categories, including Earned Leave (EL), Sick Leave (SL), Casual Leave (CL), Maternity Leave, Paternity Leave, Bereavement Leave, and Marriage Leave. There are also special unpaid leave options like Leave Without Pay (LWP) and Sabbatical Leave.\n3. User: Which of those are usually paid leaves?\n4. Assistant: The typically paid leaves include Earned Leave (EL), Sick Leave (SL), Casual Leave (CL), Maternity Leave, Paternity Leave, Bereavement Leave, and Marriage Leave. Unpaid leaves such as Leave Without Pay (LWP) and Sabbatical Leave are generally not paid.\n5. User: How should an employee apply for one of them?\n6. Assistant: The provided policy context does not specify the application procedure for leave, so I cannot provide a precise answer. Please refer to the company's HR or leave policy documents for detailed application steps.\n7. User: What happens if the leave is taken without prior approval?\n8. Assistant: The policy emphasizes that all leave requests must be approved before the absence. Taking leave without prior approval may be considered unauthorized absence, which beyond 5 consecutive working days will be treated as voluntary abandonment of employment, and may lead to disciplinary action.\n9. User: Can you summarize the key rules from this leave discussion?\n10. Assistant: Employees must apply for leave through the HRMS leave module, and manual applications are only accepted when the system is unavailable. Leave requests require approval from the Reporting Manager, Business Unit Head (for EL of 5 days or more), and HR if needed. Employees are responsible for verifying leave approval before taking leave; unauthorized leave may be considered voluntary abandonment of employment.","Turn 1\nUser: What types of leave are available to employees?\nExpected assistant answer: Employees at Meridian Technologies are provided with the following paid leave categories: Earned Leave (EL), Sick Leave (SL), Casual Leave (CL), Maternity Leave, Paternity Leave, Bereavement Leave, and Marriage Leave, as stated in clause 2 of the polic

## 10. Multi-turn SimpleCriteriaScore metrics

Only these two graded custom metrics are used for multi-turn evaluation.

In [ ]:
from ragas.metrics import SimpleCriteriaScore

multi_turn_simple_criteria_metrics = [
    SimpleCriteriaScore(
        name="multi_turn_groundedness_0_to_1",
        definition=(
            "Score from 0 to 1 how well the assistant responses are grounded in the conversation "
            "and avoid unsupported facts. Use partial credit."
        ),
        llm=evaluator_llm,
    ),
    SimpleCriteriaScore(
        name="multi_turn_answer_quality_0_to_1",
        definition=(
            "Score from 0 to 1 how factually correct, complete, context-consistent, and useful "
            "the assistant responses are. Use partial credit."
        ),
        llm=evaluator_llm,
    ),
]
print("SimpleCriteriaScore metrics:", [metric.name for metric in multi_turn_simple_criteria_metrics])

SimpleCriteriaScore metrics: ['multi_turn_groundedness_0_to_1', 'multi_turn_answer_quality_0_to_1']


In [ ]:
simple_criteria_dataset = (
    policybot_multi_turn_data["dataset_with_reference"]
    if policybot_multi_turn_data["all_have_reference"]
    else policybot_multi_turn_data["dataset_without_reference"]
)
simple_criteria_inputs_df = (
    policybot_multi_turn_parsed_df.iloc[policybot_multi_turn_data["reference_row_indexes"]].reset_index(drop=True)
    if policybot_multi_turn_data["all_have_reference"]
    else policybot_multi_turn_parsed_df.reset_index(drop=True)
)

simple_criteria_result = evaluate(
    dataset=simple_criteria_dataset,
    metrics=multi_turn_simple_criteria_metrics,
)
simple_criteria_scores = simple_criteria_result.to_pandas()
policybot_multi_turn_simple_criteria_df = pd.concat(
    [
        simple_criteria_inputs_df,
        simple_criteria_scores[ragas_score_columns(simple_criteria_scores)].reset_index(drop=True),
    ],
    axis=1,
)

print("SimpleCriteriaScore multi-turn metrics")
with pd.option_context("display.max_colwidth", None, "display.max_rows", None, "display.max_columns", None):
    display(policybot_multi_turn_simple_criteria_df)
policybot_multi_turn_simple_criteria_df.to_excel("policybot_multiturn_simple_criteria.xlsx", index=False)
print("Saved multi-turn SimpleCriteriaScore results to policybot_multiturn_simple_criteria.xlsx.")

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

SimpleCriteriaScore multi-turn metrics


,case_id,ragas_source_turn,message_count,has_reference,user_input,reference,multi_turn_groundedness_0_to_1,multi_turn_answer_quality_0_to_1
0,MULTI-000,3,6,True,"1. User: How many casual leaves can an employee take in a year?\n2. Assistant: An employee can take 6 casual leaves in a year, as per their entitlement.\n3. User: Can those leaves be carried forward if they are unused?\n4. Assistant: Yes, unused earned leave (EL) can be carried forward to the next year, subject to a maximum of 30 days as per the policy.\n5. User: What should I remember before applying for this leave?\n6. Assistant: Before applying for leave, ensure you submit your request through the HRMS leave module, verify that it is approved before starting your absence, and follow the approval workflow. Also, for leaves of 5 days or more, approval from the Business Unit Head is required.","Turn 1\nUser: How many casual leaves can an employee take in a year?\nExpected assistant answer: An employee can take 6 days of Casual Leave per calendar year.\n\nTurn 2\nUser: Can those leaves be carried forward if they are unused?\nExpected assistant answer: Yes, unused Earned Leave (EL) can be carried forward to the next calendar year, subject to a maximum accumulation of 30 days, as specified in clauses on page 4 of HR_Leave_Policy.pdf.\n\nTurn 3\nUser: What should I remember before applying for this leave?\nExpected assistant answer: Before applying for leave, remember to apply through the HRMS leave module, verify that your leave request has been approved before starting your absence, and note that manual applications are only accepted when HRMS is unavailable and must be regularized within 3 working days; also, approval depends on the workflow of Reporting Manager, Business Unit Head (for EL of 5 days or more), and HR, with the policy reviewed at least once every two years.",1,1
1,MULTI-001,5,10,True,"1. User: What types of leave are available to employees?\n2. Assistant: Employees have access to various paid leave categories, including Earned Leave (EL), Sick Leave (SL), Casual Leave (CL), Maternity Leave, Paternity Leave, Bereavement Leave, and Marriage Leave. There are also special unpaid leave options like Leave Without Pay (LWP) and Sabbatical Leave.\n3. User: Which of those are usually paid leaves?\n4. Assistant: The typically paid leaves include Earned Leave (EL), Sick Leave (SL), Casual Leave (CL), Maternity Leave, Paternity Leave, Bereavement Leave, and Marriage Leave. Unpaid leaves such as Leave Without Pay (LWP) and Sabbatical Leave are generally not paid.\n5. User: How should an employee apply for one of them?\n6. Assistant: The provided policy context does not specify the application procedure for leave, so I cannot provide a precise answer. Please refer to the company's HR or leave policy documents for detailed application steps.\n7. User: What happens if the leave is taken without prior approval?\n8. Assistant: The policy emphasizes that all leave requests must be approved before the absence. Taking leave without prior approval may be considered unauthorized absence, which beyond 5 consecutive working days will be treated as voluntary abandonment of employment, and may lead to disciplinary action.\n9. User: Can you summarize the key rules from this leave discussion?\n10. Assistant: Employees must apply for leave through the HRMS leave module, and manual applications are only accepted when the system is unavailable. Leave requests require approval from the Reporting Manager, Business Unit Head (for EL of 5 days or more), and HR if needed. Employees are responsible for verifying leave approval before taking leave; unauthorized leave may be considered voluntary abandonment of employment.","Turn 1\nUser: What types of leave are available to employees?\nExpected assistant answer: Employees at Meridian Technologies are provided with the following paid leave categories: Earned Leave (EL), Sick Leave (SL), Casual Leave (CL), Maternity Leave, Paternity Leave, Berea

Saved multi-turn SimpleCriteriaScore results to policybot_multiturn_simple_criteria.xlsx.


## 11. Output files

The notebook writes separate Excel reports for each section so you can review single-turn and multi-turn metric results independently.